# RAG 실습 — 검색→증강→생성 (수동 파이프라인) — 복원본 (대화 기반 재구성)

> ⚠️ 원본이 저장 누락으로 0바이트가 돼서, Claude와의 학습 대화에 남은 코드·구조로 **재구성**한 거야.
> 네가 그때 짠 원본 그대로는 아니니 **한 번 검토하고 직접 재실행**해. `TODO/확인` 자리는 네 데이터/환경에 맞춰 채우면 돼.
> LLM 생성이 필요한 셀은 무료 Groq API 키가 필요해 (OpenAI 크레딧 소진 상태).

## 개념
RAG = **검색(retrieval)** → **증강(augment)** → **생성(generate)**.
LLM은 컨텍스트 창 밖/학습 시점 이후 지식을 모르고, 모르면 그럴듯하게 지어냄(환각).
→ 질문과 관련된 문서를 *찾아서* 프롬프트에 넣고, 그 근거로만 답하게 함.

이 노트북은 프레임워크 없이 손으로(sentence-transformers + numpy) 흐름을 확인하는 버전.
(LangChain LCEL 버전은 `week09/rag_app.py`에 완성본이 있음.)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# 1) 지식 베이스 (작게)
docs = [
    "RAG는 관련 문서를 검색해 프롬프트에 넣고, 그것을 근거로 답을 생성한다.",
    "임베딩은 텍스트의 의미를 벡터로 바꾸며, 의미가 비슷할수록 벡터가 가깝다.",
    "파인튜닝은 모델의 가중치를 직접 학습시켜 행동이나 도메인을 바꾼다.",
    "벡터 DB는 수많은 임베딩 중 질문과 가장 가까운 것을 빠르게 찾는다.",
]

# 한국어 문서라 다국어 임베딩 모델
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
doc_emb = model.encode(docs)          # (n_docs, dim)
print("임베딩 행렬:", doc_emb.shape)

In [ ]:
def retrieve(query, k=2):
    q = model.encode([query])[0]      # (dim,)
    # TODO: 코사인 유사도 = (doc_emb · q) / (|doc_emb| * |q|)
    #   힌트: np.dot(doc_emb, q) / (np.linalg.norm(doc_emb, axis=1) * np.linalg.norm(q))
    sims = None                       # <-- 채우기
    top = np.argsort(sims)[::-1][:k]  # 유사도 높은 순 k개
    return [docs[i] for i in top]

hits = retrieve("모델 가중치를 학습시키는 방법", k=2)
print(hits)                           # 기대: 파인튜닝 관련이 위로

In [ ]:
# 3) 증강 + 생성  (Groq 무료 API 필요)
from langchain_groq import ChatGroq

context = "\n".join(retrieve("RAG가 뭐야?", k=2))
prompt = f"""아래 컨텍스트만 근거로 답해. 없으면 모른다고 해.

[컨텍스트]
{context}

[질문] RAG가 뭐야?"""

llm = ChatGroq(model="llama-3.3-70b-versatile")   # 환경변수 GROQ_API_KEY 필요
print(llm.invoke(prompt).content)
# 확인: 답이 컨텍스트 안 내용에만 근거하는지, 범위 밖 질문엔 '모른다' 하는지